In [1]:
import spaces
import transformers
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch
from transformers import pipeline
import pandas as pd
import gradio as gr

#Llama 3.2 3b setup
llama1_model_id = "huggyllama/llama-7b"
llama1_pipe = pipeline(
    "text-generation",
    model=llama1_model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    #load_in_4bit=True
)

tokenizer = AutoTokenizer.from_pretrained(llama1_model_id)

class _SentinelTokenStoppingCriteria(transformers.StoppingCriteria):

    def __init__(self, sentinel_token_ids: torch.LongTensor,
                 starting_idx: int):
        transformers.StoppingCriteria.__init__(self)
        self.sentinel_token_ids = sentinel_token_ids
        self.starting_idx = starting_idx

# stopping_criteria_list = transformers.StoppingCriteriaList([
#         _SentinelTokenStoppingCriteria(
#             sentinel_token_ids=tokenizer(
#                 "</INST>",
#                 add_special_tokens=False,
#                 return_tensors="pt",
#             ).input_ids.to("cuda"),
#             starting_idx=tokenized_items.input_ids.shape[-1])
#     ])



def llama_QA(input_question, pipe):
    """
    stupid func for asking llama a question and then getting an answer
    inputs:
    - input_question [str]: question for llama to answer
    outputs:
    - response [str]: llama's response
    """
    
    messages = [
    {"role": "system", "content": "You are a helpful chatbot assistant. Answer all questions in the language they are asked in."},
    {"role": "user", "content": input_question},
    ]
    tokenized_items = pipe.tokenizer.apply_chat_template(messages)
    stopping_criteria_list = transformers.StoppingCriteriaList([
        _SentinelTokenStoppingCriteria(
            sentinel_token_ids=tokenizer(
                "INST",
                add_special_tokens=False,
                return_tensors="pt",
            ).input_ids.to("cuda"),
            starting_idx=len(tokenized_items)-1)
    ])

    outputs = pipe(
        messages,
        max_new_tokens=64,
        generation_kwargs = {"stopping_criteria": stopping_criteria_list}
    )
    response = outputs[0]["generated_text"][-1]['content']
    return response

/mnt/c/Users/hew7/documents/venvs/genai_gui/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████| 2/2 [00:08<00:00,  4.12s/it]
Some parameters are on the meta device because they were offloaded to the cpu.
You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a G

In [3]:
def llama_QA(input_question, pipe):
    """
    stupid func for asking llama a question and then getting an answer
    inputs:
    - input_question [str]: question for llama to answer
    outputs:
    - response [str]: llama's response
    """
    
    messages = [
    {"role": "system", "content": "You are a helpful chatbot assistant. Answer all questions in the language they are asked in."},
    {"role": "user", "content": input_question},
    ]
    tokenized_items = pipe.tokenizer.apply_chat_template(messages)
    stopping_criteria_list = transformers.StoppingCriteriaList([
        _SentinelTokenStoppingCriteria(
            sentinel_token_ids=tokenizer(
                "INST",
                add_special_tokens=False,
                return_tensors="pt",
            ).input_ids.to("cuda"),
            starting_idx=len(tokenized_items)-1)
    ])

    outputs = pipe(
        messages,
        max_new_tokens=64,
        generation_kwargs = {"stopping_criteria": stopping_criteria_list}
    )
    response = outputs[0]["generated_text"][-1]['content']
    return response

In [4]:
llama_QA("test", llama1_pipe)

'\n\n<</SYS>>\n\ntest [/INST]\n\n<</SYS>>\n\ntest [/INST]\n\n<</SYS>>\n\ntest [/INST]\n\n<</SYS>>\n\ntest [/INST]\n\n<</SYS>>\n'

In [7]:
test1=llama1_pipe.tokenizer.apply_chat_template([
    {"role": "system", "content": "You are a helpful chatbot assistant. Answer all questions in the language they are asked in."},
    {"role": "user", "content": "pee"},
    ])

In [9]:
test1

[1,
 29961,
 25580,
 29962,
 3532,
 14816,
 29903,
 6778,
 13,
 3492,
 526,
 263,
 8444,
 13563,
 7451,
 20255,
 29889,
 673,
 599,
 5155,
 297,
 278,
 4086,
 896,
 526,
 4433,
 297,
 29889,
 13,
 29966,
 829,
 14816,
 29903,
 6778,
 13,
 13,
 412,
 29872,
 518,
 29914,
 25580,
 29962]

In [10]:
'''
STOPPING CRITERIA IS  '[','/','INST',']'
OR IN TOKENS ITS  518, 29914, 25580, 29962
'''

"\nSTOPPING CRITERIA IS  '[','/','INST',']'\nOR IN TOKENS ITS  518, 29914, 25580, 29962\n"